<a href="https://colab.research.google.com/github/NeDarbandsari/Facial-Emotion-Detection/blob/main/Emotion_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython import display
!pip install deepface
display.clear_output()

In [2]:
from deepface import DeepFace
import cv2
import os
from google.colab import drive

25-09-02 13:00:11 - Directory /root/.deepface has been created
25-09-02 13:00:11 - Directory /root/.deepface/weights has been created


In [3]:
video_path = "/content/drive/MyDrive/Recordings/10/"
drive.mount('/content/drive')
files = os.listdir(video_path)

Mounted at /content/drive


In [4]:
def video_emotion_detection(video, backends, alignment_modes, model,video_path):
  emotion_model = DeepFace.build_model(model)
  DeepFace.custom_models = {model: emotion_model}
  print("processing ", video)

  cap = cv2.VideoCapture(video_path+video)

  fps = cap.get(cv2.CAP_PROP_FPS)
  frame_interval = int(fps)

  prev_emotion = None
  start_time = None
  frame_count = 0
  result = {}
  result['file'] = video
  result['model'] = model
  result['backends'] = backends

  emotion=[]

  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

      # Process only every frame_interval-th frame
    if frame_count % frame_interval == 0:
        process = {}
        current_time = frame_count / fps  # current time in seconds
        try:
            # Analyze the current frame for emotion
            analysis = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False, detector_backend = backends)

            # If the result is a list (multiple faces), take the first one
            if isinstance(analysis, list):
                analysis = analysis[0]
            curr_emotion = analysis['dominant_emotion']

            # Initialize on the first processed frame
            if prev_emotion is None:
                prev_emotion = curr_emotion
                start_time = current_time
                print(f"Starting with emotion '{curr_emotion}' at {current_time:.1f}s")
                process['starting emotion'] = curr_emotion
                process['time'] = current_time
            # When the emotion changes, print the duration and update the tracker
            elif curr_emotion != prev_emotion:
                duration = current_time - start_time
                print(f"Emotion '{prev_emotion}' lasted for {duration:.1f} seconds")
                process['prev_emotion'] = prev_emotion
                process['duration'] = duration
                print(f"At {current_time:.1f}s, dominant emotion changed to '{curr_emotion}'")
                process['curr_emotion'] = curr_emotion
                process['current_time'] =current_time
                prev_emotion = curr_emotion
                start_time = current_time
            if len(process) != 0:
              emotion.append(process)
        except Exception as e:
            print(f"Error analyzing frame {frame_count}: {e}")


    frame_count += 1
  cap.release()
  # Print the duration for the final emotion
  total_time = frame_count / fps
  if prev_emotion is not None and start_time is not None:
    final_duration = total_time - start_time
    print(f"Final emotion '{prev_emotion}' lasted for {final_duration:.1f} seconds")
  print("Analysis complete.")
  result['emotions'] = emotion
  return result

In [5]:
backends = [
  'opencv',
  'ssd',
  'dlib',
  'mtcnn',
  'fastmtcnn',
  'retinaface',
  'mediapipe',
  'yolov8',
  'yolov11s',
  'yolov11n',
  'yolov11m',
  'yunet',
  'centerface',
]

alignment_modes = [True, False]

supported_models = [
    "VGG-Face",
    "Facenet",
    "Facenet512",
    "OpenFace",
    "DeepFace",
    "DeepID",
    "ArcFace",
    "Dlib",
    "SFace",
]

results=[]
for file_ in files:
  results.append(video_emotion_detection(file_, backends[3], True, supported_models[3],video_path))

25-09-02 13:00:39 - 🔗 openface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5 to /root/.deepface/weights/openface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/openface_weights.h5
To: /root/.deepface/weights/openface_weights.h5
100%|██████████| 15.3M/15.3M [00:00<00:00, 116MB/s] 


processing  03-02-10-03.mp4
25-09-02 13:00:48 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 90.2MB/s]


Starting with emotion 'sad' at 0.0s
Emotion 'sad' lasted for 1.0 seconds
At 1.0s, dominant emotion changed to 'neutral'
Emotion 'neutral' lasted for 1.0 seconds
At 2.0s, dominant emotion changed to 'sad'
Emotion 'sad' lasted for 1.0 seconds
At 3.0s, dominant emotion changed to 'neutral'
Final emotion 'neutral' lasted for 2.6 seconds
Analysis complete.
processing  04-02-10-03.mp4
Starting with emotion 'neutral' at 0.0s
Final emotion 'neutral' lasted for 5.7 seconds
Analysis complete.
processing  03-01-10-03.mp4
Starting with emotion 'neutral' at 0.0s
Final emotion 'neutral' lasted for 4.0 seconds
Analysis complete.
processing  02-02-10-03.mp4
Starting with emotion 'angry' at 0.0s
Emotion 'angry' lasted for 1.0 seconds
At 1.0s, dominant emotion changed to 'neutral'
Final emotion 'neutral' lasted for 2.9 seconds
Analysis complete.
processing  01-02-10-03.mp4
Starting with emotion 'neutral' at 0.0s
Emotion 'neutral' lasted for 3.0 seconds
At 3.0s, dominant emotion changed to 'angry'
Emotio

In [6]:
results

[{'file': '03-02-10-03.mp4',
  'model': 'OpenFace',
  'backends': 'mtcnn',
  'emotions': [{'starting emotion': 'sad', 'time': 0.0},
   {'prev_emotion': 'sad',
    'duration': 1.0,
    'curr_emotion': 'neutral',
    'current_time': 1.0},
   {'prev_emotion': 'neutral',
    'duration': 1.0,
    'curr_emotion': 'sad',
    'current_time': 2.0},
   {'prev_emotion': 'sad',
    'duration': 1.0,
    'curr_emotion': 'neutral',
    'current_time': 3.0}]},
 {'file': '04-02-10-03.mp4',
  'model': 'OpenFace',
  'backends': 'mtcnn',
  'emotions': [{'starting emotion': 'neutral', 'time': 0.0}]},
 {'file': '03-01-10-03.mp4',
  'model': 'OpenFace',
  'backends': 'mtcnn',
  'emotions': [{'starting emotion': 'neutral', 'time': 0.0}]},
 {'file': '02-02-10-03.mp4',
  'model': 'OpenFace',
  'backends': 'mtcnn',
  'emotions': [{'starting emotion': 'angry', 'time': 0.0},
   {'prev_emotion': 'angry',
    'duration': 1.0,
    'curr_emotion': 'neutral',
    'current_time': 1.0}]},
 {'file': '01-02-10-03.mp4',
  '